# Grade 12 · Unit 4 — Overfitting

> The data are generated here: a smooth curve plus noise (**SYNTHETIC**).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

rng = np.random.default_rng(4)
def make(n):
    x = np.sort(rng.uniform(-1, 1, n))
    return x, np.sin(2.5 * x) + rng.normal(0, 0.25, n)
x_tr, y_tr = make(25)
x_te, y_te = make(200)
plt.scatter(x_tr, y_tr, label="train"); plt.scatter(x_te, y_te, s=6, alpha=0.3, label="test"); plt.legend(); plt.show()

## Fit polynomials of increasing degree
**Predict:** will test error keep falling as the degree rises?

In [ ]:
mse = lambda a, b: float(np.mean((a - b) ** 2))
rows = []
for degree in (1, 2, 3, 5, 8, 12):
    coef = np.polyfit(x_tr, y_tr, degree)
    rows.append((degree, round(mse(np.polyval(coef, x_tr), y_tr), 4), round(mse(np.polyval(coef, x_te), y_te), 4)))
res = pd.DataFrame(rows, columns=["degree", "train_mse", "test_mse"])
assert res["train_mse"].is_monotonic_decreasing      # nested models: training error can only fall
print(res)

## Picture the over-fit

In [ ]:
grid = np.linspace(-1, 1, 300)
for d in (3, 12):
    plt.plot(grid, np.polyval(np.polyfit(x_tr, y_tr, d), grid), label=f"degree {d}")
plt.scatter(x_tr, y_tr, c="k", s=12); plt.ylim(-2.5, 2.5); plt.legend(); plt.show()

**Make:** which degree would you choose using validation data only? Why is picking by *training* error a mistake?